## Graph Features for Fraud Detection

Fraud rings often look normal per-transaction — what's abnormal is the *structure*: a tight mutual cluster, or unusual connectivity for the account type. Tabular, per-transaction features can't represent network position at all; graph topology features can.

**Topology Features: Degree, PageRank, Clustering Coefficient**

1. Nodes = accounts, edges = transactions — this directly models money flow, so centrality/community features map cleanly to "unusually connected in the flow of funds."
2. Degree alone doesn't cleanly separate a fraud ring from a legitimate high-degree hub (like a payroll processor) — a ring has degree ~9–11, a hub has degree 62, but degree alone isn't the discriminator.
3. Clustering coefficient is the real discriminator: a fraud ring is a dense mutual cluster (neighbors also connected to each other), while a star-shaped hub has clustering coefficient exactly 0, since its spokes never transact with each other.

In [ ]:
import numpy as np
import networkx as nx
import pandas as pd

RNG = np.random.default_rng(42)
N = 1000

G = nx.barabasi_albert_graph(n=N, m=2, seed=42)   # realistic hub-and-spoke backbone

ring_nodes = list(range(500, 508))                # fraud ring: dense near-clique
for i in ring_nodes:
    for j in ring_nodes:
        if i != j:
            G.add_edge(i, j)

hub_node = 999                                    # legitimate hub: star, spokes don't interconnect
hub_spokes = RNG.choice([n for n in range(N) if n != hub_node and n not in ring_nodes], size=60, replace=False)
for s in hub_spokes:
    G.add_edge(hub_node, s)

degree = dict(G.degree())
pagerank = nx.pagerank(G, alpha=0.85)
clustering = nx.clustering(G)
topology_df = pd.DataFrame({
    "account_id": list(G.nodes()),
    "degree": [degree[n] for n in G.nodes()],
    "pagerank": [pagerank[n] for n in G.nodes()],
    "clustering_coeff": [clustering[n] for n in G.nodes()],
})

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print("\nRing nodes (fraud):")
print(topology_df[topology_df.account_id.isin(ring_nodes)][["account_id", "degree", "clustering_coeff"]])
print("\nHub node (legitimate, high-degree):")
print(topology_df[topology_df.account_id == hub_node][["account_id", "degree", "clustering_coeff"]])

**Choosing Node/Edge Definitions — and Their Blind Spot**

1. Nodes = accounts, edges = transactions directly models money flow — but a ring that avoids direct transactions while sharing a device fingerprint or IP won't be connected by transaction edges at all.
2. Fix: add a second, shared-attribute edge type (device/IP) to catch that case — coordinated accounts hiding behind transaction-level separation become directly linkable once a second relationship type is added.

In [ ]:
G_device = G.copy()
# Two accounts NOT directly transacting, but sharing a device ID (a device-fraud pattern transaction edges miss)
device_pair = (ring_nodes[0] + 150, ring_nodes[1] + 150)
dist_before = nx.shortest_path_length(G_device, *device_pair)

G_device.add_edge(*device_pair)   # the shared-device edge
dist_after = nx.shortest_path_length(G_device, *device_pair)

print(f"Accounts {device_pair}: shortest path via transactions only = {dist_before}")
print(f"Accounts {device_pair}: shortest path after adding a shared-device edge = {dist_after}")

**Confirming the Ring Is Tabularly Invisible**

1. Train a tabular-only baseline alongside the combined (tabular + graph) model.
2. If tabular-only scores near the random baseline on ring detection, that's direct evidence the ring has no tabular signature by construction — adding graph features unlocks detection at all here, not an incremental improvement on an already-working baseline.

In [ ]:
is_fraud = topology_df["account_id"].isin(ring_nodes).astype(int)
avg_txn_amount = RNG.lognormal(mean=7, sigma=1, size=N)
account_age_days = RNG.integers(30, 3000, size=N)
kyc_risk_score = RNG.uniform(0, 1, size=N)

full_df = topology_df.copy()
full_df["avg_txn_amount"] = avg_txn_amount
full_df["account_age_days"] = account_age_days
full_df["kyc_risk_score"] = kyc_risk_score
full_df["is_fraud"] = is_fraud

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

tabular_cols = ["avg_txn_amount", "account_age_days", "kyc_risk_score"]
graph_cols = ["degree", "pagerank", "clustering_coeff"]

X_train, X_test, y_train, y_test = train_test_split(
    full_df[tabular_cols + graph_cols], full_df["is_fraud"], test_size=0.3, stratify=full_df["is_fraud"], random_state=42
)

def fit_eval(cols, label):
    pos_rate = y_train.mean()
    m = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                       scale_pos_weight=(1 - pos_rate) / pos_rate, eval_metric="logloss", random_state=42)
    m.fit(X_train[cols], y_train)
    proba = m.predict_proba(X_test[cols])[:, 1]
    print(f"{label}: PR-AUC = {average_precision_score(y_test, proba):.3f}")
    return m, proba

_, proba_tabular = fit_eval(tabular_cols, "Tabular-only baseline")
model_combined, proba_combined = fit_eval(tabular_cols + graph_cols, "Tabular + graph topology")

**Why the False-Positive Rate Stays Stable**

1. The usual precision/recall tradeoff happens when more recall comes from loosening the threshold on the *same* signal.
2. Graph features are orthogonal — a structurally different population, not a stronger version of the tabular signal — so the false-positive rate on the original tabular population doesn't move.
3. The PR-AUC gap from the previous cell (tabular-only near-chance, combined much higher) is the evidence: the combined model isn't pushing more borderline tabular cases over threshold, it's separating a population tabular features never scored highly at all.

**Validating the Signal Is Real, Not a Graph Artifact**

1. Direct check: does the legitimate hub score low despite high degree, because its clustering coefficient is 0?
2. Pair the score with SHAP attribution — "this account sits in a tightly connected cluster with a 0.58 clustering coefficient" is investigable; "score = 0.87" is not.

In [ ]:
hub_row = full_df[full_df.account_id == hub_node]
hub_proba = model_combined.predict_proba(hub_row[tabular_cols + graph_cols])[0, 1]
print(f"Legitimate hub (degree={hub_row.degree.values[0]}, clustering={hub_row.clustering_coeff.values[0]:.3f}) "
      f"-> fraud probability: {hub_proba:.3f}")

ring_rows = full_df[full_df.account_id.isin(ring_nodes)]
ring_proba = model_combined.predict_proba(ring_rows[tabular_cols + graph_cols])[:, 1]
print(f"Fraud ring accounts -> mean fraud probability: {ring_proba.mean():.3f}")

import shap
explainer = shap.TreeExplainer(model_combined)
sv = explainer.shap_values(X_test)
mean_abs_shap = pd.Series(np.abs(sv).mean(axis=0), index=X_test.columns).sort_values(ascending=False)
print("\nMean |SHAP| by feature:")
print(mean_abs_shap)

**Cold Start — A New Account With No History**

1. Degree and clustering are literally zero/undefined for a node with no edges — the model doesn't fail, but has no graph signal to work with, so scoring falls back entirely on whatever tabular features exist (which may also be thin for a new account).
2. A near-zero score for a zero-history account isn't necessarily safe — it means the model has no opinion, not that the account is verified clean. New accounts need a separate onboarding-risk control (KYC-based, not graph-based) until enough transaction history accumulates for graph features to mean anything.

In [ ]:
G_cold = G.copy()
new_node = 1000
G_cold.add_node(new_node)   # zero transactions yet

new_degree = G_cold.degree(new_node)
new_clustering = nx.clustering(G_cold, new_node)
new_pagerank = nx.pagerank(G_cold, alpha=0.85)[new_node]
print(f"New account: degree={new_degree}, clustering={new_clustering}, pagerank={new_pagerank:.6f}")

cold_start_row = pd.DataFrame([{
    "avg_txn_amount": avg_txn_amount.mean(), "account_age_days": 0, "kyc_risk_score": 0.5,
    "degree": new_degree, "pagerank": new_pagerank, "clustering_coeff": new_clustering,
}])
cold_start_proba = model_combined.predict_proba(cold_start_row[tabular_cols + graph_cols])[0, 1]
print(f"Model's fraud probability for a brand-new, zero-history account: {cold_start_proba:.3f}")

**What's Still Missing for a Production Version**

- Real temporal graph, not a synthetic add-on: a production system needs genuine edge-timestamp data feeding continuously — a low-degree "stealth" ring evades static clustering entirely, which is why a temporal feature (edge-formation rate) matters as a defense.
- Scale: full-graph PageRank/clustering don't scale to a bank's daily transaction volume — needs subgraph scoping and incremental updates in production.
- Shared-attribute edges: only transaction edges exist here; a ring hiding behind device/IP sharing needs that second edge type built into the real graph, not just demonstrated as a toggle.
- Cold-start onboarding control: new accounts need a non-graph-based risk control until transaction history accumulates.